# Atlas 自动向量化重嵌入验证

这个 Notebook 用于验证 Atlas `autoEmbed` 在文档更新后的自动重嵌入行为。

验证流程：
1. 初始化一组商品样例数据。
2. 创建 `vectorSearch` 索引，并对 `description` 使用 `autoEmbed`。
3. 使用“原始意图”进行向量检索。
4. 更新同一商品的描述为“新意图”。
5. 轮询检索结果，直到反映最新描述。

In [ ]:
# 如有需要，每次内核启动后先执行一次。
%pip install -r requirements.txt

In [ ]:
import os
import time
from datetime import datetime, timezone

import certifi
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel

In [ ]:
load_dotenv()

MONGODB_ATLAS_URI = os.getenv("MONGODB_ATLAS_URI")
DB_NAME = os.getenv("ATLAS_DB_NAME", "atlas_auto_embed_demo")
BASE_COLL_NAME = os.getenv("ATLAS_CASE_COLLECTION_NAME", "products_auto_reembed_case")
RUN_SUFFIX = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
COLL_NAME = f"{BASE_COLL_NAME}_{RUN_SUFFIX}"
INDEX_NAME = "auto_embed_description_idx"

if not MONGODB_ATLAS_URI:
    raise ValueError("请先在 .env 中配置 MONGODB_ATLAS_URI 再运行。")

print(f"数据库: {DB_NAME}")
print(f"集合: {COLL_NAME}")
print(f"索引: {INDEX_NAME}")

In [ ]:
client = MongoClient(MONGODB_ATLAS_URI, tlsCAFile=certifi.where())
db = client[DB_NAME]
coll = db[COLL_NAME]

print(f"使用新的测试集合: {COLL_NAME}")

In [ ]:
TARGET_SKU = "SKU-HEADPHONE-001"

initial_target_description = (
    "无线头戴式主动降噪耳机，"
    " 柔软耳罩，40 小时续航，适合通勤和长途飞行。"
)

updated_target_description = (
    "不锈钢保温水壶，带防漏杯盖，"
    " 可保冷 24 小时，适合徒步和健身。"
)

seed_docs = [
    {
        "sku": TARGET_SKU,
        "name": "AeroSound X1",
        "category": "electronics",
        "description": initial_target_description,
        "updatedAt": datetime.now(timezone.utc),
    },
    {
        "sku": "SKU-COFFEE-001",
        "name": "Single-Origin Coffee Beans",
        "category": "grocery",
        "description": "中度烘焙咖啡豆，带有巧克力风味和轻微柑橘余韵。",
        "updatedAt": datetime.now(timezone.utc),
    },
    {
        "sku": "SKU-LAMP-001",
        "name": "DeskGlow LED Lamp",
        "category": "home",
        "description": "可调节 LED 台灯，触控操作，支持暖光到冷光切换。",
        "updatedAt": datetime.now(timezone.utc),
    },
    {
        "sku": "SKU-YOGA-001",
        "name": "FlexGrip Yoga Mat",
        "category": "fitness",
        "description": "防滑瑜伽垫，额外缓冲设计，适合居家训练和拉伸。",
        "updatedAt": datetime.now(timezone.utc),
    },
    {
        "sku": "SKU-BACKPACK-001",
        "name": "TrailPack 30L Backpack",
        "category": "outdoors",
        "description": "轻量徒步背包，附带防雨罩和多个装备分区。",
        "updatedAt": datetime.now(timezone.utc),
    },
]

coll.insert_many(seed_docs)
print(f"已插入 {coll.count_documents({})} 条样例文档。")

In [ ]:
index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "autoEmbed",
                "modality": "text",
                "path": "description",
                "model": "voyage-4-lite",
                "numDimensions": 1024,
            },
            {"type": "filter", "path": "sku"},
            {"type": "filter", "path": "category"},
        ]
    },
    name=INDEX_NAME,
    type="vectorSearch",
)

existing_index_names = {idx["name"] for idx in coll.list_search_indexes()}
if INDEX_NAME in existing_index_names:
    coll.drop_search_index(INDEX_NAME)

created = coll.create_search_index(model=index_model)
print(f"已创建索引: {created}")

def wait_for_index_ready(collection, name, timeout_s=300, interval_s=5):
    start = time.time()
    while True:
        idx = next((i for i in collection.list_search_indexes() if i["name"] == name), None)
        if idx and idx.get("status") == "READY" and idx.get("queryable", True):
            return idx
        if time.time() - start > timeout_s:
            raise TimeoutError(f"索引 {name} 在 {timeout_s} 秒内未就绪")
        time.sleep(interval_s)

wait_for_index_ready(coll, INDEX_NAME)
print("索引状态 READY，可查询。")

In [ ]:
def vector_search(query_text, limit=3):
    pipeline = [
        {
            "$vectorSearch": {
                "index": INDEX_NAME,
                "path": "description",
                "query": {"text": query_text},
                "numCandidates": 50,
                "limit": limit,
            }
        },
        {
            "$project": {
                "_id": 0,
                "sku": 1,
                "name": 1,
                "category": 1,
                "description": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        },
    ]
    return list(coll.aggregate(pipeline))

def print_hits(title, results):
    print(title)
    for rank, doc in enumerate(results, start=1):
        print(f"{rank}. {doc['sku']} | score={doc['score']:.4f} | {doc['name']}")
    print()

def find_rank_and_score(results, sku):
    for rank, doc in enumerate(results, start=1):
        if doc["sku"] == sku:
            return rank, doc["score"]
    return None, None

In [ ]:
old_intent_query = "适合坐飞机使用的无线降噪耳机"
new_intent_query = "适合徒步使用的不锈钢保温水壶"

before_old = vector_search(old_intent_query)
before_new = vector_search(new_intent_query)

print_hits("更新前 -> 旧意图查询", before_old)
print_hits("更新前 -> 新意图查询", before_new)

In [ ]:
update_result = coll.update_one(
    {"sku": TARGET_SKU},
    {
        "$set": {
            "description": updated_target_description,
            "updatedAt": datetime.now(timezone.utc),
        }
    },
)

print(f"匹配文档: {update_result.matched_count}, 修改文档: {update_result.modified_count}")
print("已在 MongoDB 文档中更新目标商品描述。")

In [ ]:
timeout_s = 240
poll_every_s = 5
start = time.time()

print("开始轮询 Atlas 自动重嵌入结果...")

while True:
    current_old = vector_search(old_intent_query)
    current_new = vector_search(new_intent_query)

    old_rank, old_score = find_rank_and_score(current_old, TARGET_SKU)
    new_rank, new_score = find_rank_and_score(current_new, TARGET_SKU)

    elapsed = int(time.time() - start)
    print(
        f"+{elapsed:>3}s | 旧意图 rank={old_rank} score={old_score} | "
        f"新意图 rank={new_rank} score={new_score}"
    )

    if new_rank == 1 and (old_rank is None or old_rank > 1):
        print("检测到重嵌入完成：商品已匹配更新后的新意图。")
        break

    if time.time() - start > timeout_s:
        raise TimeoutError(
            "等待重嵌入刷新超时，可增加 timeout 或重试。"
        )

    time.sleep(poll_every_s)

In [ ]:
after_old = vector_search(old_intent_query)
after_new = vector_search(new_intent_query)

print_hits("更新后 -> 旧意图查询", after_old)
print_hits("更新后 -> 新意图查询", after_new)

print("验证完成：向量检索已反映最新商品描述。")

### 可选清理

如需删除测试数据和索引，请取消注释后执行。

In [ ]:
# coll.drop_search_index(INDEX_NAME)
# db.drop_collection(COLL_NAME)